> - **Setup and Widgets**

In [0]:
from pyspark.sql.functions import (
    col, explode, current_timestamp, lit, input_file_name
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
)

dbutils.widgets.text("storage_account", "retailprojec1data")
dbutils.widgets.text("container", "bronze")
dbutils.widgets.text("directory", "transactions")

storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
directory = dbutils.widgets.get("directory")

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
source_path = f"abfss://transactions@retailprojec1data.dfs.core.windows.net/raw/"
checkpoint_path = f"{base_path}/{directory}/_checkpoints/"
schema_location = f"{base_path}/{directory}/_schema/"
target_table_path = f"{base_path}/transactions_delta/"

print(f"Source:      {source_path}")
print(f"Target:      {target_table_path}")
print(f"Checkpoint:  {checkpoint_path}")

**- Defining the Schema**

In [0]:

transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("register_id", IntegerType(), True),
    StructField("cashier_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("transaction_time", StringType(), True),  
    StructField("payment_method", StringType(), True),
    StructField("product_sku", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("status", StringType(), True),
])


file_schema = StructType([
    StructField("data", ArrayType(transaction_schema), True),
    StructField("pagination", StringType(), True),  
    StructField("window", StringType(), True),
])

print("Schema defined")

**- Read with Auto Loader**

In [0]:
raw_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "false")  
    .schema(file_schema)
    .option("multiLine", "true")  
    .load(source_path))

print("Stream defined. Schema:")
raw_stream.printSchema()

**Transforming : Explode the data Array**

In [0]:
transactions_stream = (raw_stream
    .select(
        explode(col("data")).alias("txn"),
        col("_metadata.file_path").alias("source_file")  
    )
    .select(
        col("txn.*"),                                   
        col("source_file"),
        current_timestamp().alias("ingestion_ts"),
        lit("pos_api_rest").alias("source_system")
    ))

print("Transformations applied. Output schema:")
transactions_stream.printSchema()

**Write to bronze Delta**

In [0]:
query = (transactions_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)  # process all currently-available files then stop
    .start(target_table_path))

query.awaitTermination()

print("✅ Auto Loader run complete")

**Verifying output**

In [0]:
# Get the number of rows written in this run
progress = query.lastProgress
input_rows = progress.get("sources", [{}])[0].get("numInputRows", 0) if progress else 0

# Read total Bronze count for context
total_rows = spark.read.format("delta").load(target_table_path).count()

print(f"This run processed {input_rows} new rows")
print(f"Total rows in Bronze Delta: {total_rows}")

# Clean exit message for monitoring
dbutils.notebook.exit(f"processed_{input_rows}_total_{total_rows}")